#### 130. List	the	emps	with	Hire	date	in	format	June	4,	1988.

In [0]:
use sql_catalog.sql_schema

In [0]:
select ename, date_format(hiredate, 'MM-dd-yyyy') as hiredate 
from emp;

ename,hiredate
SMITH,12-17-1980
ALLEN,02-20-1981
WARD,02-22-1981
JONES,04-02-1981
MARTIN,09-28-1981
BLAKE,05-01-1981
CLARK,06-09-1981
SCOTT,12-09-1982
KING,11-17-1981
TURNER,09-08-1981


#### 131. Print	a	list	of	emp’s	Listing	‘just	salary’	if	Salary	is	more	than	1500,	on target	if	Salary	is	1500	and	‘Below	1500’	if	Salary	is	less	than	1500.

In [0]:
select ename, 
  case
    when sal > 1500 then 'just salary'
    when sal = 1500 then 'on target'
    when sal < 1500 then '1500'
    --else 'nothing'
  end as sal_status
from emp

ename,sal_status
SMITH,1500
ALLEN,just salary
WARD,1500
JONES,just salary
MARTIN,1500
BLAKE,just salary
CLARK,just salary
SCOTT,just salary
KING,just salary
TURNER,on target


#### 132 Write	a	query	which	return	the	day	of	the	week	for	any	date	entered	in format	‘DD-MM-YY’.

In [0]:
select ename, dayofweek(hiredate) as dayofweek 
from emp

ename,dayofweek
SMITH,4
ALLEN,6
WARD,1
JONES,5
MARTIN,2
BLAKE,6
CLARK,3
SCOTT,5
KING,3
TURNER,3


#### 133.	Write	a	query	to	calculate	the	length	of	service	of	any	employee	with	the company

In [0]:
SELECT ename, hiredate, datediff(current_date(), hiredate) AS days_of_service
FROM emp;

ename,hiredate,days_of_service
SMITH,1980-12-17,16724
ALLEN,1981-02-20,16659
WARD,1981-02-22,16657
JONES,1981-04-02,16618
MARTIN,1981-09-28,16439
BLAKE,1981-05-01,16589
CLARK,1981-06-09,16550
SCOTT,1982-12-09,16002
KING,1981-11-17,16389
TURNER,1981-09-08,16459


#### 135. Emps	hired	on	or	before	15th	of	any	month	are	paid	on	the	last	Friday	of that	month	those	hired	after	15th	are	paid	on	the	first	Friday	of	the	following month.	Print	a	list	of	emps	their	hire	date	and	the	first	pay	date.	Sort	on	hire	date.

In [0]:
SELECT
  e.ename,
  date_format(e.hiredate, 'yyyy-MM-dd') AS hire_date,
  date_format(
    CASE
      WHEN dayofmonth(e.hiredate) <= 15
        THEN next_day(date_add(last_day(e.hiredate), -7), 'FRI')
      ELSE next_day(date_add(date_add(last_day(e.hiredate), 1), -1), 'FRI')
    END,
    'yyyy-MM-dd'
   ) AS first_pay_date
FROM Emp e
WHERE e.hiredate IS NOT NULL

/*
last_day(date): gives the last day of the month for date.
date_add(date, n): adds n days (can be negative to subtract).
next_day(date, 'FRI'): returns the next Friday strictly after date.

Goal: Get the last Friday of the same month.
Why it works

last_day(e.hiredate) → last calendar day of that month (e.g., if hiredate = 2025-12-10, last_day = 2025-12-31).
date_add(..., -7) → go back one week to 2025-12-24.
next_day(..., 'FRI') → find the next Friday after 2025-12-24, which is 2025-12-26.

Because we started one week before month end, the “next Friday” we hit is guaranteed to be the final Friday of that month.
Mini example

Month end: Dec 31, 2025 (Wednesday)
Back 7 days: Dec 24, 2025 (Wednesday)
Next Friday after Dec 24: Dec 26, 2025 → last Friday of Dec 2025 ✅
*/

#### 36. Count	the	no.	of	characters	with	out	considering	spaces	for	each	name.

In [0]:
select ename,length(replace(ename, " ", "")) as char_count 
from emp;

ename,char_count
SMITH,5
ALLEN,5
WARD,4
JONES,5
MARTIN,6
BLAKE,5
CLARK,5
SCOTT,5
KING,4
TURNER,6


#### 137. Find	out	the	emps	who	are	getting	decimal	value	in	their	Sal	without	using like	operator.

In [0]:
select ename from emp 
where sal is not null
and cast(sal as string) like '%.%'

ename


In [0]:

-- Employees whose salary has a fractional (decimal) part
SELECT
  ename,
  sal
FROM Emp
WHERE sal IS NOT NULL
  AND MOD(sal, 1) <> 0;      -- fractional part exists


ename,sal


#### 138. List	those	emps	whose	Salary	contains	first	four	digit	of	their	Deptno.

In [0]:
select e.ename 
from emp e join dept d on e.deptno = d.deptno
where e.sal in (select e.sal from emp e where e.sal = substring(d.deptno, 1,4))

ename


In [0]:

SELECT
  e.ename,
  e.sal,
  d.deptno
FROM Emp e
JOIN Dept d
  ON e.deptno = d.deptno
WHERE REPLACE(CAST(e.sal AS STRING), '.', '') LIKE CONCAT(SUBSTRING(CAST(d.deptno AS STRING), 1, 4), '%');

/*

Convert both sal and deptno to strings.
Take the first 4 characters of deptno (if it has fewer than 4, this just takes all its digits).
Check if the salary string contains that substring.
*/

#### 138. List those emps whose Salary contains last four digit of their Deptno.

In [0]:

-- Employees whose salary contains the last 4 digits of their deptno
SELECT
  e.ename,
  e.sal,
  d.deptno
FROM Emp e
JOIN Dept d
  ON e.deptno = d.deptno
WHERE REPLACE(CAST(e.sal AS STRING), '.', '') LIKE CONCAT('%', RIGHT(CAST(d.deptno AS STRING), 4), '%');

/*
RIGHT(CAST(d.deptno AS STRING), 4) → last four digits of deptno.
REPLACE(CAST(e.sal AS STRING), '.', '') → salary digits without the decimal point.
LIKE '%…%' → checks if those four digits appear anywhere in the salary

*/

#### 139. List	those	Managers	who	are	getting	less	than	his	emps	Salary.

In [0]:
select m.ename as mgrname, e.ename as empname, e.sal, m.sal
from emp e join emp m
on e.mgr = m.empno
where m.sal < e.sal

mgrname,empname,sal,sal
JONES,SCOTT,3000,2975
JONES,FORD,3000,2975


#### 140. Print	the	details	of	all	the	emps	who	are	sub-ordinates	to	Blake.

In [0]:

-- Direct reports of Blake
SELECT e.*
FROM Emp e
JOIN Emp m
  ON e.mgr = m.empno
WHERE m.ename = 'BLAKE';  --


empno,ename,job,mgr,hiredate,sal,comm,deptno
7499,ALLEN,SALESMAN,7698,1981-02-20,1600,300,30
7521,WARD,SALESMAN,7698,1981-02-22,1250,500,30
7654,MARTIN,SALESMAN,7698,1981-09-28,1250,1400,30
7844,TURNER,SALESMAN,7698,1981-09-08,1500,0,30
7900,JAMES,CLERK,7698,1981-12-03,950,null,30


#### 41. List	the	emps	who	are	working	as	Managers	using	co-related	sub-query.

In [0]:
select e.ename as empname, m.ename as manager_name
from emp e join emp m 
on e.mgr = m.empno 
where m.ename = 'mgr'

In [0]:
select e.ename as empname, m.ename as manager_name
from emp e join emp m 
on e.mgr = m.empno 
where m.ename in (select e.ename from emp e where e.ename = 'mgr')

In [0]:
select	*	from	emp	where	empno	in	(select	mgr	from	emp)

#### 42.	List	the	emps	whose	Mgr	name	is	‘Jones’	and	also	with	his	Manager name.

In [0]:
select e.ename as empname, m.ename as manager_name
from emp e join emp m 
on e.mgr = m.empno
where m.ename = "JONES"

empname,manager_name
SCOTT,JONES
FORD,JONES


#### 144. Find	out	how	may	Managers	are	their	in	the	company.

In [0]:
select count(*) from emp where job = 'MANAGER'

--OR
--select count(*) from emp where emopno in (select mgr from emp);

count(*)
3


#### 145. Find	Average	salary	and	Average	total	remuneration	for	each	Job	type. Remember	Salesman	earn	commission.secommm

In [0]:
select job, avg(sal) as avg_Sal, 
avg(case 
  when upper(job) = 'SALESMAN' THEN sal + coalesce(comm, 0)
  else sal
  end) as avg_sal from emp
group by job
order by job

#### 46. Check	whether	all	the	emps	numbers	are	indeed	unique.

In [0]:
select empno, count(*) from emp group by empno having count(*)>1;

In [0]:
SELECT
  CASE
    WHEN EXISTS (
      SELECT empno
      FROM Emp
      GROUP BY empno
      HAVING COUNT(*) > 1
    ) THEN 'Duplicates Found'
    ELSE 'All empnos are unique'
  END AS result

#### 147. List	the	emps	who	are	drawing	less	than	1000	Sort	the	output	by	Salary.

In [0]:
select empno 
from emp 
where empno in(
    select empno 
    from emp 
    where sal<1000 
    order by sal asc)

empno
7369
7900


In [0]:
select empno 
    from emp 
    where sal<1000 
    order by sal asc

empno
7369
7900


In [0]:
select empno from emp where sal < 1000 order by empno asc;

#### 148. List	the	employee	Name,	Job,	Annual	Salary,	deptno,	Dept	name	and grade	who	earn	36000	a	year	or	who	are	not	CLERKS.

In [0]:
SELECT e.ename,
       e.job,
       d.dname,
       (12*e.sal) as annual_sal,
       d.deptno,
       s.grade
FROM emp e
JOIN dept d ON e.deptno = d.deptno
JOIN salgrade s ON e.sal BETWEEN s.losal AND s.hisal
where (12*e.sal) = '36000'
or 
e.job <> 'CLAARK'

ename,job,dname,annual_sal,deptno,grade
BLAKE,MANAGER,SALES,34200,30,4
ADAMS,CLERK,RESEARCH,13200,20,1
WARD,SALESMAN,SALES,15000,30,2
FORD,ANALYST,RESEARCH,36000,20,4
SMITH,CLERK,RESEARCH,9600,20,1
JONES,MANAGER,RESEARCH,35700,20,4
SCOTT,ANALYST,RESEARCH,36000,20,4
MILLER,CLERK,ACCOUNTING,15600,10,2
ALLEN,SALESMAN,SALES,19200,30,3
MARTIN,SALESMAN,SALES,15000,30,2


#### 149. Find	out	the	Job	that	was	filled	in	the	first	half	of	1983	and	same	job	that was	filled	during	the	same	period	of	1984.

In [0]:
select job, hiredate
from emp
where hiredate between date('1983-01-01') and date('1983-01-30')

job,hiredate
CLERK,1983-01-12


#### 150. Find	out	the	emps	who	joined	in	the	company	before	their	Managers

In [0]:
select e.ename as empname, m.ename as mgr_name, e.hiredate, m.hiredate
from emp e join emp m 
on e.mgr = m.empno
where e.hiredate <  m.hiredate;

empname,mgr_name,hiredate,hiredate
SMITH,FORD,1980-12-17,1981-12-03
ALLEN,BLAKE,1981-02-20,1981-05-01
WARD,BLAKE,1981-02-22,1981-05-01
JONES,KING,1981-04-02,1981-11-17
BLAKE,KING,1981-05-01,1981-11-17
CLARK,KING,1981-06-09,1981-11-17
